In [1]:
%%capture
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
!pip uninstall mamba-ssm causal-conv1d torch torchvision torchaudio -y
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip cache purge
MAX_JOBS=1
!pip install causal-conv1d>=1.4.0 --no-cache-dir --no-build-isolation --no-binary :all:
MAX_JOBS=1
!pip install mamba-ssm --no-cache-dir --no-build-isolation --no-binary :all: 

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import json
import math
import random                             
import pickle
import shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from datetime import datetime
from collections import Counter
from copy import deepcopy

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F            
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast       

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision.transforms as transforms

from mamba_ssm import Mamba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()

print(f'Device: {device}')
print(f'Available GPUs: {num_gpus}')
if torch.cuda.is_available():
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory // 1024**3} GB VRAM')

Device: cuda
Available GPUs: 2
  GPU 0: Tesla T4 | 14 GB VRAM
  GPU 1: Tesla T4 | 14 GB VRAM


In [4]:
DATA_ROOT = '/kaggle/input/datasets/devil1696/noisy-dronrf-kaggle-spectrogram'

CLASS_NAMES = sorted(['DJI', 'FutabaT14', 'FutabaT7', 'Graupner', 'Noise', 'Taranis', 'Turnigy'])
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

print('Class registry:')
for idx, name in IDX_TO_CLASS.items():
    print(f'  {idx}: {name}')
print(f'\nTotal classes: {NUM_CLASSES}')

@dataclass
class Config:
    img_size:          int   = 224
    patch_size:        int   = 4
    in_channels:       int   = 3
    num_classes:       int   = NUM_CLASSES
    embed_dim:         int   = 96
    depth:             int   = 4
    d_state:           int   = 16
    d_conv:            int   = 4
    expand:            int   = 2
    drop_rate:         float = 0.3
    drop_path_rate:    float = 0.1
    layer_scale_init:  float = 0.1
    batch_size:        int   = 8
    learning_rate:     float = 2e-4
    weight_decay:      float = 0.001
    epochs:            int   = 100
    patience:          int   = 25
    grad_clip_norm:    float = 1.0
    label_smoothing:   float = 0.1
    warmup_epochs:     int   = 15
    val_split:         float = 0.20
    random_state:      int   = 42


config = Config()

print(f'\nConfig loaded')
print(f'  embed_dim={config.embed_dim}  depth={config.depth}  d_state={config.d_state}')
print(f'  lr={config.learning_rate}  epochs={config.epochs}  batch_size={config.batch_size}')


Class registry:
  0: DJI
  1: FutabaT14
  2: FutabaT7
  3: Graupner
  4: Noise
  5: Taranis
  6: Turnigy

Total classes: 7

Config loaded
  embed_dim=96  depth=4  d_state=16
  lr=0.0002  epochs=100  batch_size=8


In [5]:
def parse_snr(filename: str) -> int:
    """
    Extract SNR integer from filename.
    Examples: IQdata_sample1150_target0_snr-12.png  -> -12
              IQdata_sample135_target0_snr0.png      ->   0
    """
    stem = Path(filename).stem
    match = re.search(r'snr(-?\d+)$', stem)
    return int(match.group(1)) if match else 0


def load_dataset(data_root: str) -> Tuple[List[str], np.ndarray, np.ndarray]:
    """Load all image paths, integer class labels, and SNR values."""
    paths, labels, snrs = [], [], []
    print(f'\nLoading dataset from: {data_root}')
    for class_name in CLASS_NAMES:
        class_dir = Path(data_root) / class_name
        if not class_dir.exists():
            print(f'  WARNING: Missing folder: {class_dir}')
            continue
        png_files = sorted(class_dir.glob('*.png'))
        lbl = CLASS_TO_IDX[class_name]
        for p in png_files:
            paths.append(str(p))
            labels.append(lbl)
            snrs.append(parse_snr(p.name))
        print(f'  {class_name:<12}: {len(png_files):5d} images')

    labels = np.array(labels, dtype=np.int64)
    snrs   = np.array(snrs,   dtype=np.int32)
    print(f'\nTotal: {len(paths)} images')
    print(f'SNR range: {snrs.min()} dB to {snrs.max()} dB')
    print(f'All SNR levels: {sorted(set(snrs.tolist()))} dB')
    return paths, labels, snrs


all_paths, all_labels, all_snrs = load_dataset(DATA_ROOT)

indices = np.arange(len(all_paths))
train_idx, val_idx = train_test_split(
    indices,
    test_size=config.val_split,
    random_state=config.random_state,
    stratify=all_labels,
)
train_idx = sorted(train_idx.tolist())
val_idx   = sorted(val_idx.tolist())

train_paths  = [all_paths[i]  for i in train_idx]
train_labels = all_labels[train_idx]
train_snrs   = all_snrs[train_idx]

val_paths    = [all_paths[i]  for i in val_idx]
val_labels   = all_labels[val_idx]
val_snrs     = all_snrs[val_idx]

ALL_SNR_LEVELS = sorted(set(all_snrs.tolist()))

print(f'\nTrain: {len(train_paths)} | Val (held-out 20%): {len(val_paths)}')
print(f'Train class counts: {dict(Counter(train_labels.tolist()))}')
print(f'Val   class counts: {dict(Counter(val_labels.tolist()))}')
print(f'SNR levels in val split: {sorted(set(val_snrs.tolist()))} dB')


Loading dataset from: /kaggle/input/datasets/devil1696/noisy-dronrf-kaggle-spectrogram
  DJI         :  1280 images
  FutabaT14   :  3472 images
  FutabaT7    :   801 images
  Graupner    :   801 images
  Noise       :  8872 images
  Taranis     :  1663 images
  Turnigy     :   855 images

Total: 17744 images
SNR range: -20 dB to 30 dB
All SNR levels: [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30] dB

Train: 14195 | Val (held-out 20%): 3549
Train class counts: {0: 1024, 1: 2778, 2: 641, 3: 641, 4: 7097, 5: 1330, 6: 684}
Val   class counts: {0: 256, 1: 694, 2: 160, 3: 160, 4: 1775, 5: 333, 6: 171}
SNR levels in val split: [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30] dB


In [6]:

# ── Step 1: Compute dataset-specific normalisation stats ──────────────────────
def compute_spectrogram_stats(paths: List[str],
                               sample_size: int = 2000) -> Tuple[List[float], List[float]]:
    """
    Compute per-channel GLOBAL mean and std by collecting every pixel
    from `sample_size` training images into one large tensor.

    Uses global pixel statistics (not per-image stats) — this is what
    torchvision.transforms.Normalize actually expects. Per-image std
    on flat-coloured spectrograms is ~0.05 which causes ±12 input blowup.
    """
    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(paths), min(sample_size, len(paths)), replace=False)
    to_t = transforms.ToTensor()

    pixel_lists = [[] for _ in range(3)]  

    for i in idx:
        img = to_t(Image.open(paths[i]).convert('RGB'))  
        for c in range(3):
            pixel_lists[c].append(img[c].reshape(-1))  

    mean, std = [], []
    for c in range(3):
        all_pixels = torch.cat(pixel_lists[c])          
        mean.append(all_pixels.mean().item())
        std.append(all_pixels.std().item())

    return mean, std


print('Computing spectrogram normalisation statistics from training set...')
SPEC_MEAN, SPEC_STD = compute_spectrogram_stats(train_paths, sample_size=2000)
print(f'  SPEC_MEAN = {[round(v, 4) for v in SPEC_MEAN]}')
print(f'  SPEC_STD  = {[round(v, 4) for v in SPEC_STD]}')


# ── Step 2: SpecAugment — valid augmentation for spectrograms ─────────────────
class SpecAugment:
    """
    Randomly mask contiguous frequency bands (rows) and time frames (columns).
    Applied AFTER ToTensor+Normalize. Masked value = 0.0 (normalised mean).

    freq_mask_param : max frequency bins to mask per mask
    time_mask_param : max time frames to mask per mask
    num_freq_masks  : number of independent frequency masks
    num_time_masks  : number of independent time masks
    """

    def __init__(self,
                 freq_mask_param: int = 20,
                 time_mask_param: int = 20,
                 num_freq_masks:  int = 2,
                 num_time_masks:  int = 2):
        self.F  = freq_mask_param
        self.T  = time_mask_param
        self.nF = num_freq_masks
        self.nT = num_time_masks

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        _, H, W = x.shape

        for _ in range(self.nF):
            f  = random.randint(0, self.F)
            f0 = random.randint(0, max(H - f, 0))
            x[:, f0:f0 + f, :] = 0.0

        for _ in range(self.nT):
            t  = random.randint(0, self.T)
            t0 = random.randint(0, max(W - t, 0))
            x[:, :, t0:t0 + t] = 0.0

        return x


# ── Step 3: Transform pipelines ───────────────────────────────────────────────
def get_train_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),  
        SpecAugment(freq_mask_param=20,                       
                    time_mask_param=20,
                    num_freq_masks=2,
                    num_time_masks=2),
    ])


def get_val_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),  
    ])


# ── Step 4: Dataset & DataLoader ──────────────────────────────────────────────
class DroneRFDataset(Dataset):
    """Single-task spectrogram dataset with class labels and SNR metadata."""

    def __init__(self, paths: List[str], labels: np.ndarray,
                 snrs: np.ndarray, transform=None):
        self.paths     = paths
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.snrs      = snrs
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def build_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    """Inverse-frequency sampler to handle class imbalance."""
    class_counts = Counter(labels.tolist())
    weights = [1.0 / class_counts[int(lb)] for lb in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)


def build_dataloaders(cfg: Config,
                      tr_paths, tr_labels, tr_snrs,
                      vl_paths, vl_labels, vl_snrs,
                      num_workers: int = 2):
    train_ds = DroneRFDataset(tr_paths, tr_labels, tr_snrs,
                              transform=get_train_transforms(cfg.img_size))
    val_ds   = DroneRFDataset(vl_paths, vl_labels, vl_snrs,
                              transform=get_val_transforms(cfg.img_size))

    sampler = build_weighted_sampler(tr_labels)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              sampler=sampler, num_workers=num_workers,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True)
    return train_loader, val_loader


# ── Sanity check ──────────────────────────────────────────────────────────────
_sanity_train_loader, _sanity_val_loader = build_dataloaders(
    config,
    train_paths, train_labels, train_snrs,
    val_paths,   val_labels,   val_snrs,
)

print(f'Train batches : {len(_sanity_train_loader)}')
print(f'Val   batches : {len(_sanity_val_loader)}')

imgs, lbls = next(iter(_sanity_train_loader))
print(f'Batch image shape : {imgs.shape}')
print(f'Batch label shape : {lbls.shape}')
print(f'Batch label sample: {lbls[:8].tolist()}')
print(f'Batch image range : [{imgs.min():.3f}, {imgs.max():.3f}]')

del _sanity_train_loader, _sanity_val_loader


Computing spectrogram normalisation statistics from training set...
  SPEC_MEAN = [0.2995, 0.6544, 0.4342]
  SPEC_STD  = [0.0992, 0.081, 0.0623]
Train batches : 1775
Val   batches : 444
Batch image shape : torch.Size([8, 3, 224, 224])
Batch label shape : torch.Size([8])
Batch label sample: [0, 3, 6, 0, 4, 2, 3, 2]
Batch image range : [-7.885, 7.063]


In [7]:
class StochasticDepth(nn.Module):
    """Drop entire residual paths randomly during training (stochastic depth)."""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand   = torch.rand(shape, dtype=x.dtype, device=x.device)
        rand   = torch.floor(rand + keep_prob)
        return (x / keep_prob) * rand


class LayerScale(nn.Module):
    """Learned per-channel scaling; critical for training ViT-style models from scratch."""

    def __init__(self, dim: int, init_value: float = 0.1):
        super().__init__()
        self.gamma = nn.Parameter(init_value * torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.gamma


class PatchEmbed(nn.Module):
    """Strided convolution that converts an image to a sequence of patch embeddings."""

    def __init__(self, img_size: int, patch_size: int,
                 in_chans: int, embed_dim: int):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, N, D)
        return self.proj(x).flatten(2).transpose(1, 2)


class VimBlock(nn.Module):
    """
    Bidirectional Vision Mamba block.
    Forward and backward SSM scans are summed, scaled by LayerScale,
    then added to the residual via a stochastic depth gate.
    """

    def __init__(self, dim: int, d_state: int, d_conv: int, expand: int,
                 drop_path: float = 0.0, layer_scale_init: float = 0.1):
        super().__init__()
        self.norm       = nn.LayerNorm(dim)
        self.mamba_fwd  = Mamba(d_model=dim, d_state=d_state,
                                d_conv=d_conv, expand=expand)
        self.mamba_bwd  = Mamba(d_model=dim, d_state=d_state,
                                d_conv=d_conv, expand=expand)
        self.layer_scale = LayerScale(dim, layer_scale_init)
        self.drop_path   = StochasticDepth(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.norm(x)
        y_fwd = self.mamba_fwd(h)
        y_bwd = self.mamba_bwd(h.flip([1])).flip([1])
        out   = self.layer_scale(y_fwd + y_bwd)
        return residual + self.drop_path(out)


class VisionMamba(nn.Module):
    """
    Vision Mamba for single-task classification.

    Architecture:
        PatchEmbed  (B,3,224,224) -> (B,196,D)
        CLS token prepend          -> (B,197,D)
        Positional embedding added
        VimBlock x depth (bidirectional SSM per block)
        LayerNorm + CLS head       -> (B, num_classes)
    """

    def __init__(self, cfg: Config):
        super().__init__()
        D = cfg.embed_dim

        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size,
                                      cfg.in_channels, D)
        N = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, D))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, D))
        self.pos_drop  = nn.Dropout(p=cfg.drop_rate)
        dpr = [x.item() for x in
               torch.linspace(0, cfg.drop_path_rate, cfg.depth)]

        self.layers = nn.ModuleList([
            VimBlock(
                dim=D,
                d_state=cfg.d_state,
                d_conv=cfg.d_conv,
                expand=cfg.expand,
                drop_path=dpr[i],
                layer_scale_init=cfg.layer_scale_init,
            )
            for i in range(cfg.depth)
        ])

        self.norm = nn.LayerNorm(D)
        self.head = nn.Sequential(
            nn.Dropout(p=cfg.drop_rate),
            nn.Linear(D, cfg.num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embed)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x[:, 0])


_test_cfg = Config()
_model    = VisionMamba(_test_cfg).to(device)
_inp      = torch.randn(2, 3, 224, 224, device=device)
_out      = _model(_inp)
print(f'VisionMamba output shape: {_out.shape}')
total_p  = sum(p.numel() for p in _model.parameters())
train_p  = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Parameters: {total_p:,} total | {train_p:,} trainable')
del _model, _inp, _out
torch.cuda.empty_cache()

VisionMamba output shape: torch.Size([2, 7])
Parameters: 853,255 total | 853,255 trainable


In [8]:
def adjust_lr(optimizer: torch.optim.Optimizer,
              epoch: int, cfg: Config) -> float:
    """Linear warmup then cosine annealing."""
    if epoch < cfg.warmup_epochs:
        lr = cfg.learning_rate * (epoch + 1) / cfg.warmup_epochs
    else:
        progress = (epoch - cfg.warmup_epochs) / max(
            cfg.epochs - cfg.warmup_epochs, 1)
        lr = cfg.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr


def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    num_classes: int) -> Dict[str, float]:
    """Accuracy, weighted-F1, macro-FPR, macro-FNR."""
    acc = accuracy_score(y_true, y_pred) * 100.0
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100.0
    cm  = confusion_matrix(y_true, y_pred, labels=range(num_classes))

    fpr_list, fnr_list = [], []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
        fnr_list.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)

    return {
        'accuracy': round(acc, 2),
        'f1':       round(f1,  2),
        'fpr':      round(np.mean(fpr_list) * 100.0, 2),
        'fnr':      round(np.mean(fnr_list) * 100.0, 2),
    }


print('LR schedule and metrics utilities defined')

LR schedule and metrics utilities defined


In [9]:
# ── FocalLoss ─────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss — down-weights easy examples so training focuses on hard ones
    (FutabaT14 confused with Noise, DJI confused with FutabaT7/Noise).

    FL(pt) = -(1 - pt)^gamma * log(pt)

    gamma=2.0 is the standard setting.
    Label smoothing is applied before computing pt.
    """

    def __init__(self, gamma: float = 2.0,
                 label_smoothing: float = 0.1,
                 num_classes: int = NUM_CLASSES):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.num_classes     = num_classes

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        # Build smoothed target distribution
        with torch.no_grad():
            smooth_targets = torch.full_like(
                logits, self.label_smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing)

        log_prob = F.log_softmax(logits, dim=-1)
        prob     = log_prob.exp()
        ce = -(smooth_targets * log_prob).sum(dim=-1)         
        pt = prob.gather(1, targets.unsqueeze(1)).squeeze(1)    
        return (((1.0 - pt) ** self.gamma) * ce).mean()


# ── train_one_epoch ───────────────────────────────────────────────────────────
def train_one_epoch(model: nn.Module,
                    loader: DataLoader,
                    criterion: nn.Module,
                    optimizer: torch.optim.Optimizer,
                    cfg: Config) -> Tuple[float, float]:   
    model.train()
    total_loss    = 0.0
    total_correct = 0
    total_samples = 0
    nan_batches   = 0

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(dtype=torch.bfloat16):              
            logits = model(images)
            loss   = criterion(logits, labels)
        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            optimizer.zero_grad(set_to_none=True)
            if nan_batches <= 3:
                print(f'  WARNING: NaN/Inf loss at step {step}. Skipping batch.')
            continue

        loss.backward()                                  
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=cfg.grad_clip_norm)
        optimizer.step()                                  

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += bs

    if nan_batches > 0:
        print(f'  [nan_batches total this epoch: {nan_batches}]')

    if total_samples == 0:
        return float('nan'), 0.0
    return total_loss / total_samples, total_correct / total_samples * 100.0


# ── evaluate ──────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model: nn.Module,
             loader: DataLoader,
             criterion: nn.Module) -> Tuple[float, Dict]:
    model.eval()
    total_loss    = 0.0
    total_samples = 0
    all_preds     = []
    all_labels    = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(dtype=torch.bfloat16):             
            logits = model(images)
            loss   = criterion(logits, labels)

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(
        np.array(all_labels), np.array(all_preds), NUM_CLASSES)
    return total_loss / total_samples, metrics


# ── run_training ──────────────────────────────────────────────────────────────
def run_training(cfg: Config,
                 tr_paths, tr_labels, tr_snrs,
                 vl_paths, vl_labels, vl_snrs,
                 tag: str = 'vim',
                 verbose: bool = True) -> Dict:
    """
    Full training loop: warmup + cosine LR, bfloat16 AMP,
    gradient clipping, focal loss, early stopping.
    Returns dict with best state, metrics, history.
    """
    train_loader, val_loader = build_dataloaders(
        cfg, tr_paths, tr_labels, tr_snrs,
        vl_paths, vl_labels, vl_snrs)

    model = VisionMamba(cfg)
    if num_gpus > 1:
        model = nn.DataParallel(model)
        if verbose:
            print(f'  Using DataParallel on {num_gpus} GPUs')
    model = model.to(device)

    criterion = FocalLoss(                                
        gamma=2.0,
        label_smoothing=cfg.label_smoothing,
        num_classes=cfg.num_classes,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  [],
        'lr': [],
    }
    best_val_acc = 0.0
    best_metrics = None
    best_state   = None
    patience_ctr = 0

    if verbose:
        print(f'\n{"="*70}')
        print(f'Training: {tag.upper()}')
        print(f'  embed_dim={cfg.embed_dim}  depth={cfg.depth}  '
              f'd_state={cfg.d_state}  expand={cfg.expand}')
        print(f'  lr={cfg.learning_rate}  drop_path={cfg.drop_path_rate}  '
              f'layer_scale={cfg.layer_scale_init}')
        print(f'  epochs={cfg.epochs}  patience={cfg.patience}  '
              f'warmup={cfg.warmup_epochs}')
        print(f'  loss=FocalLoss(gamma=2.0)  amp=bfloat16')
        print(f'{"="*70}')

    for epoch in range(cfg.epochs):
        lr = adjust_lr(optimizer, epoch, cfg)
        history['lr'].append(lr)

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, cfg) 
        val_loss, val_metrics = evaluate(model, val_loader, criterion)
        val_acc = val_metrics['accuracy']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_metrics = deepcopy(val_metrics)
            patience_ctr = 0
            raw_model_init = model.module if isinstance(model, nn.DataParallel) else model
            best_state     = deepcopy(raw_model_init.state_dict())
        else:
            patience_ctr += 1

        if verbose and (epoch % 5 == 0 or
                        epoch == cfg.epochs - 1 or
                        patience_ctr == 0):
            print(
                f'  Epoch {epoch+1:3d}/{cfg.epochs} | '
                f'LR {lr:.6f} | '
                f'Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | '
                f'Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | '
                f'Best {best_val_acc:.2f}%'
            )

        if patience_ctr >= cfg.patience:
            if verbose:
                print(f'  Early stopping at epoch {epoch+1}')
            break

    if verbose:
        print(f'\nBest Val Acc: {best_val_acc:.2f}%')
        print(f'  Acc={best_metrics["accuracy"]:.2f}%  '
              f'F1={best_metrics["f1"]:.2f}%  '
              f'FPR={best_metrics["fpr"]:.2f}%  '
              f'FNR={best_metrics["fnr"]:.2f}%')

    del model, criterion, optimizer      
    torch.cuda.empty_cache()

    return {
        'tag':          tag,
        'cfg_snapshot': deepcopy(cfg),
        'best_val_acc': best_val_acc,
        'best_metrics': best_metrics,
        'best_state':   best_state,
        'history':      history,
    }


print('FocalLoss, training and evaluation functions defined')

FocalLoss, training and evaluation functions defined


In [10]:
result = run_training(
    config,
    train_paths, train_labels, train_snrs,
    val_paths,   val_labels,   val_snrs,
    tag='vim_updated', verbose=True,
)
print(f'Best Val Acc: {result["best_val_acc"]:.2f}%')
print(f'Metrics: {result["best_metrics"]}')

os.makedirs('vim_outputs', exist_ok=True)
torch.save(result['best_state'], 'vim_outputs/vim_best.pth')
print('Weights saved to vim_outputs/vim_best.pth')

final_history = result['history']

  Using DataParallel on 2 GPUs

Training: VIM_UPDATED
  embed_dim=96  depth=4  d_state=16  expand=2
  lr=0.0002  drop_path=0.1  layer_scale=0.1
  epochs=100  patience=25  warmup=15
  loss=FocalLoss(gamma=2.0)  amp=bfloat16
  Epoch   1/100 | LR 0.000013 | Train Loss 1.4459 Acc 14.55% | Val Loss 1.4154 Acc 9.44% | Best 9.44%
  Epoch   2/100 | LR 0.000027 | Train Loss 1.4412 Acc 14.29% | Val Loss 1.3908 Acc 30.23% | Best 30.23%
  Epoch   6/100 | LR 0.000080 | Train Loss 1.4112 Acc 17.41% | Val Loss 1.3854 Acc 28.43% | Best 30.23%
  Epoch   8/100 | LR 0.000107 | Train Loss 1.3962 Acc 18.89% | Val Loss 1.3657 Acc 35.59% | Best 35.59%
  Epoch  11/100 | LR 0.000147 | Train Loss 1.3798 Acc 20.73% | Val Loss 1.4457 Acc 24.94% | Best 35.59%
  Epoch  12/100 | LR 0.000160 | Train Loss 1.3773 Acc 20.87% | Val Loss 1.3061 Acc 38.46% | Best 38.46%
  Epoch  16/100 | LR 0.000200 | Train Loss 1.3342 Acc 23.98% | Val Loss 1.3013 Acc 34.54% | Best 38.46%
  Epoch  21/100 | LR 0.000198 | Train Loss 1.1146 A

In [11]:
@torch.no_grad()
def evaluate_at_snr(model: nn.Module,
                    paths: List[str],
                    labels: np.ndarray,
                    snrs: np.ndarray,
                    snr_value: int,
                    cfg: Config) -> Optional[Dict]:
    mask       = snrs == snr_value
    snr_paths  = [paths[i] for i in range(len(paths)) if mask[i]]
    snr_labels = labels[mask]

    if len(snr_paths) == 0:
        return None

    ds     = DroneRFDataset(snr_paths, snr_labels, snrs[mask],
                            transform=get_val_transforms(cfg.img_size))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    model.eval()

    all_preds, all_true = [], []
    for images, lbl in loader:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.bfloat16):            
            logits = model(images)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_true.extend(lbl.tolist())

    return compute_metrics(np.array(all_true), np.array(all_preds), NUM_CLASSES)


# Load best model
eval_model = VisionMamba(config).to(device)
eval_model.load_state_dict(result['best_state'])
eval_model.eval()

_, eval_val_loader = build_dataloaders(
    config,
    train_paths, train_labels, train_snrs,
    val_paths,   val_labels,   val_snrs,
)

print(f'{"="*80}')
print('SNR ROBUSTNESS EVALUATION  (held-out 20% val split)')
print(f'{"="*80}')
print(f'{"SNR (dB)":>10} {"N":>6} {"Accuracy":>10} {"F1":>10} '
      f'{"FPR":>8} {"FNR":>8}')
print('-' * 60)

snr_robustness = {}

for snr_val in ALL_SNR_LEVELS:
    result = evaluate_at_snr(
        eval_model, val_paths, val_labels, val_snrs, snr_val, config)

    if result is None:
        print(f'  {snr_val:>+5} dB  :  no samples in val split')
        continue

    snr_robustness[snr_val] = result
    n = int((val_snrs == snr_val).sum())

    print(f'  {snr_val:>+5} dB  '
          f'  n={n:>5}  '
          f'  Acc={result["accuracy"]:>5.1f}%  '
          f'  F1={result["f1"]:>5.1f}%  '
          f'  FPR={result["fpr"]:>5.1f}%  '
          f'  FNR={result["fnr"]:>5.1f}%')

eval_criterion = FocalLoss(                            
    gamma=2.0,
    label_smoothing=config.label_smoothing,
    num_classes=config.num_classes,
).to(device)

_, overall_metrics = evaluate(eval_model, eval_val_loader, eval_criterion)

print(f'\n{"OVERALL (all SNRs)":>10}  '
      f'  n={len(val_paths):>5}  '
      f'  Acc={overall_metrics["accuracy"]:>5.1f}%  '
      f'  F1={overall_metrics["f1"]:>5.1f}%  '
      f'  FPR={overall_metrics["fpr"]:>5.1f}%  '
      f'  FNR={overall_metrics["fnr"]:>5.1f}%')

SNR ROBUSTNESS EVALUATION  (held-out 20% val split)
  SNR (dB)      N   Accuracy         F1      FPR      FNR
------------------------------------------------------------
    -20 dB    n=  159    Acc= 52.8%    F1= 45.8%    FPR= 10.4%    FNR= 62.6%
    -18 dB    n=  132    Acc= 57.6%    F1= 54.6%    FPR= 10.5%    FNR= 61.7%
    -16 dB    n=  133    Acc= 62.4%    F1= 58.3%    FPR=  9.1%    FNR= 53.0%
    -14 dB    n=  134    Acc= 61.9%    F1= 58.0%    FPR=  8.3%    FNR= 50.0%
    -12 dB    n=  133    Acc= 66.9%    F1= 64.6%    FPR=  7.6%    FNR= 44.4%
    -10 dB    n=  122    Acc= 72.1%    F1= 70.9%    FPR=  5.8%    FNR= 33.3%
     -8 dB    n=  141    Acc= 82.3%    F1= 81.8%    FPR=  3.4%    FNR= 22.6%
     -6 dB    n=  137    Acc= 82.5%    F1= 82.7%    FPR=  3.4%    FNR= 17.8%
     -4 dB    n=  160    Acc= 90.6%    F1= 90.7%    FPR=  1.6%    FNR= 11.8%
     -2 dB    n=  134    Acc= 81.3%    F1= 82.5%    FPR=  3.2%    FNR= 31.9%
     +0 dB    n=  149    Acc= 88.6%    F1= 89.0%    FPR=  1

In [12]:
@torch.no_grad()
def per_class_accuracy(model: nn.Module,
                        paths: List[str],
                        labels: np.ndarray,
                        cfg: Config) -> Tuple[Dict[str, float], np.ndarray]:
    ds     = DroneRFDataset(paths, labels, np.zeros(len(labels), dtype=np.int32),
                            transform=get_val_transforms(cfg.img_size))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    model.eval()
    all_preds, all_true = [], []
    for images, lbl in loader:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.bfloat16):              
            logits = model(images)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_true.extend(lbl.tolist())

    preds  = np.array(all_preds)
    truths = np.array(all_true)
    cm     = confusion_matrix(truths, preds, labels=range(NUM_CLASSES))
    per_class = {}
    for i, name in enumerate(CLASS_NAMES):
        total   = cm[i, :].sum()
        correct = cm[i, i]
        per_class[name] = round(correct / total * 100.0, 2) if total > 0 else 0.0
    return per_class, cm


per_class, cm = per_class_accuracy(eval_model, val_paths, val_labels, config)

print(f'{"="*50}')
print('PER-CLASS ACCURACY (val split)')
print(f'{"="*50}')
for name, acc in per_class.items():
    bar = '#' * int(acc / 5)
    print(f'  {name:<12}: {acc:>5.1f}%  {bar}')

print(f'\nConfusion Matrix (rows=True, cols=Pred):')
header = '       ' + ''.join(f'{n[:6]:>7}' for n in CLASS_NAMES)
print(header)
for i, name in enumerate(CLASS_NAMES):
    row = f'{name[:6]:<7}' + ''.join(f'{cm[i,j]:>7}' for j in range(NUM_CLASSES))
    print(row)

PER-CLASS ACCURACY (val split)
  DJI         :  69.9%  #############
  FutabaT14   :  73.3%  ##############
  FutabaT7    :  76.9%  ###############
  Graupner    :  59.4%  ###########
  Noise       :  86.2%  #################
  Taranis     :  91.3%  ##################
  Turnigy     :  93.6%  ##################

Confusion Matrix (rows=True, cols=Pred):
           DJI Futaba Futaba Graupn  Noise Tarani Turnig
DJI        179      9     10      2     51      5      0
Futaba       4    509     26      6    127      1     21
Futaba       4     12    123      1     16      1      3
Graupn       2      3      1     95      7     52      0
Noise       83     95     21     25   1530     17      4
Tarani       5      1      0     12     11    304      0
Turnig       0      4      4      1      2      0    160


In [13]:
# Save model, weights, results and artifacts

OUTPUT_DIR = 'vim_drone_rf_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save(
    eval_model.state_dict(),
    f'{OUTPUT_DIR}/vim_best_weights.pth'
)
print('Saved vim_best_weights.pth')

torch.save({
    'model_name': 'VisionMamba',
    'cfg': {
        'embed_dim':        config.embed_dim,
        'depth':            config.depth,
        'd_state':          config.d_state,
        'd_conv':           config.d_conv,
        'expand':           config.expand,
        'drop_rate':        config.drop_rate,
        'drop_path_rate':   config.drop_path_rate,
        'layer_scale_init': config.layer_scale_init,
        'num_classes':      config.num_classes,
        'img_size':         config.img_size,
        'patch_size':       config.patch_size,
    },
    'class_names':  CLASS_NAMES,
    'best_val_acc': overall_metrics['accuracy'],
    'state_dict':   eval_model.state_dict(),
}, f'{OUTPUT_DIR}/vim_checkpoint.pth')
print('Saved vim_checkpoint.pth')

results_json = {
    'model':       'VisionMamba',
    'dataset':     'Noisy Drone RF v2 Spectrogram',
    'timestamp':   datetime.now().isoformat(),
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'architecture': {
        'embed_dim':        config.embed_dim,
        'depth':            config.depth,
        'd_state':          config.d_state,
        'expand':           config.expand,
        'drop_path_rate':   config.drop_path_rate,
        'layer_scale_init': config.layer_scale_init,
    },
    'training': {
        'best_lr':       config.learning_rate,
        'batch_size':    config.batch_size,
        'warmup_epochs': config.warmup_epochs,
        'loss_fn':       'FocalLoss(gamma=2.0)',
        'amp_dtype':     'bfloat16',
        'normalization': {
            'mean': SPEC_MEAN,
            'std':  SPEC_STD,
        },
    },
    'overall_val_metrics': overall_metrics,
    'per_class_accuracy':  per_class,
    'snr_robustness': {
        str(snr): m for snr, m in snr_robustness.items()
    },
}

with open(f'{OUTPUT_DIR}/vim_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('Saved vim_results.json')

with open(f'{OUTPUT_DIR}/vim_history.pkl', 'wb') as f:
    pickle.dump(final_history, f)
print('Saved vim_history.pkl')

zip_name = f'vim_drone_rf_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f'ZIP: {zip_name}.zip')
print(f'Contents: {sorted(os.listdir(OUTPUT_DIR))}')

try:
    from IPython.display import FileLink, display
    display(FileLink(f'{zip_name}.zip'))
except Exception:
    pass

print(f'\n{"="*70}')
print('ALL ARTIFACTS SAVED')
print(f'  Accuracy : {overall_metrics["accuracy"]:.2f}%')
print(f'  F1 Score : {overall_metrics["f1"]:.2f}%')
print(f'  FPR      : {overall_metrics["fpr"]:.2f}%')
print(f'  FNR      : {overall_metrics["fnr"]:.2f}%')
print(f'{"="*70}')

Saved vim_best_weights.pth
Saved vim_checkpoint.pth
Saved vim_results.json
Saved vim_history.pkl
ZIP: vim_drone_rf_20260220_210652.zip
Contents: ['vim_best_weights.pth', 'vim_checkpoint.pth', 'vim_history.pkl', 'vim_results.json']


/kaggle/working/vim_drone_rf_20260220_210652.zip


ALL ARTIFACTS SAVED
  Accuracy : 81.71%
  F1 Score : 81.68%
  FPR      : 3.68%
  FNR      : 21.35%


In [14]:
# Load best model for all post-training analyses

eval_model = VisionMamba(config).to(device)
eval_model.load_state_dict(
    torch.load('/kaggle/working/vim_outputs/vim_best.pth', map_location=device, weights_only=True)
)
eval_model.eval()

os.makedirs('vim_drone_rf_output', exist_ok=True)

@torch.no_grad()
def get_val_preds_and_probs(model):
    ds     = DroneRFDataset(val_paths, val_labels, val_snrs,
                            transform=get_val_transforms(config.img_size))
    loader = DataLoader(ds, batch_size=config.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, probs_list = [], []
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.bfloat16):
            logits = model(images)
        p = F.softmax(logits.float(), dim=1)
        preds.extend(logits.argmax(1).cpu().tolist())
        probs_list.append(p.cpu())
    return np.array(preds), torch.cat(probs_list, dim=0).numpy()

val_preds, val_probs = get_val_preds_and_probs(eval_model)
true_labels = val_labels.copy()

acc = (val_preds == true_labels).mean() * 100.0
print(f'Model accuracy on val set: {acc:.2f}%')
print(f'Val samples: {len(true_labels)}')


Model accuracy on val set: 81.71%
Val samples: 3549


In [15]:
# Analysis 1: McNemar's Test  (model vs majority-class baseline)

from statsmodels.stats.contingency_tables import mcnemar

majority_class   = int(np.bincount(train_labels).argmax())
preds_majority   = np.full(len(true_labels), majority_class, dtype=np.int64)

correct_model    = (val_preds    == true_labels).astype(int)
correct_majority = (preds_majority == true_labels).astype(int)

a = int(np.sum((correct_model == 1) & (correct_majority == 1)))
b = int(np.sum((correct_model == 1) & (correct_majority == 0)))
c = int(np.sum((correct_model == 0) & (correct_majority == 1)))
d = int(np.sum((correct_model == 0) & (correct_majority == 0)))

table_mc  = np.array([[a, b], [c, d]])
result_mc = mcnemar(table_mc, exact=False, correction=True)

print('McNemar Test: VisionMamba-B  vs  Majority-Class Baseline')
print(f'  Both correct           : {a}')
print(f'  Model correct, base wrong: {b}')
print(f'  Model wrong, base correct: {c}')
print(f'  Both wrong             : {d}')
print(f'  Chi-squared            : {result_mc.statistic:.4f}')
print(f'  p-value                : {result_mc.pvalue:.2e}')
if result_mc.pvalue < 0.05:
    print('  Significant improvement over majority baseline (alpha=0.05)')
else:
    print('  No significant difference at alpha=0.05')


McNemar Test: VisionMamba-B  vs  Majority-Class Baseline
  Both correct           : 1530
  Model correct, base wrong: 1370
  Model wrong, base correct: 245
  Both wrong             : 404
  Chi-squared            : 782.2762
  p-value                : 3.85e-172
  Significant improvement over majority baseline (alpha=0.05)


In [16]:
# Analysis 2: Cochran's Q Test  (model vs majority vs uniform-random baseline)

from statsmodels.stats.contingency_tables import cochrans_q

rng_rand = np.random.default_rng(42)
preds_random    = rng_rand.integers(0, NUM_CLASSES, size=len(true_labels))
correct_random  = (preds_random == true_labels).astype(int)

table_q  = np.column_stack([correct_model, correct_majority, correct_random])
result_q = cochrans_q(table_q)

print('Cochran Q Test: [VisionMamba-B,  Majority Baseline,  Random Baseline]')
print(f'  Q statistic      : {result_q.statistic:.4f}')
print(f'  p-value          : {result_q.pvalue:.2e}')
print(f'  VisionMamba-B acc: {correct_model.mean()*100:.2f}%')
print(f'  Majority acc     : {correct_majority.mean()*100:.2f}%  (class={CLASS_NAMES[majority_class]})')
print(f'  Random acc       : {correct_random.mean()*100:.2f}%')
if result_q.pvalue < 0.05:
    print('  Significant performance difference among classifiers (alpha=0.05)')
else:
    print('  No significant difference among classifiers at alpha=0.05')


Cochran Q Test: [VisionMamba-B,  Majority Baseline,  Random Baseline]
  Q statistic      : 2892.4617
  p-value          : 0.00e+00
  VisionMamba-B acc: 81.71%
  Majority acc     : 50.01%  (class=Noise)
  Random acc       : 13.78%
  Significant performance difference among classifiers (alpha=0.05)


In [17]:
# Analysis 3: Bootstrap Confidence Intervals  (1000 iterations)

def bootstrap_ci(y_true, y_pred, metric_fn, n_iter=1000, ci=95, seed=42):
    rng   = np.random.default_rng(seed)
    n     = len(y_true)
    stats = np.zeros(n_iter)
    for i in range(n_iter):
        idx      = rng.integers(0, n, size=n)
        stats[i] = metric_fn(y_true[idx], y_pred[idx])
    alpha = (100 - ci) / 2
    return stats.mean(), np.percentile(stats, alpha), np.percentile(stats, 100 - alpha)

def acc_fn(yt, yp):
    return (yt == yp).mean() * 100.0

def f1_fn(yt, yp):
    return f1_score(yt, yp, average='weighted', zero_division=0) * 100.0

def fpr_fn(yt, yp):
    cm = confusion_matrix(yt, yp, labels=range(NUM_CLASSES))
    vals = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - cm[i, :].sum() - fp
        vals.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    return np.mean(vals) * 100.0

def fnr_fn(yt, yp):
    cm = confusion_matrix(yt, yp, labels=range(NUM_CLASSES))
    vals = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        vals.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)
    return np.mean(vals) * 100.0

print('Bootstrap Confidence Intervals  (VisionMamba-B, 95% CI, n=1000)')
print('-' * 60)
for name, fn in [('Accuracy', acc_fn), ('Weighted F1', f1_fn),
                 ('Macro FPR', fpr_fn), ('Macro FNR', fnr_fn)]:
    mean, lo, hi = bootstrap_ci(true_labels, val_preds, fn)
    print(f'  {name:<14}: {mean:6.2f}%  [{lo:.2f}%, {hi:.2f}%]')


Bootstrap Confidence Intervals  (VisionMamba-B, 95% CI, n=1000)
------------------------------------------------------------
  Accuracy      :  81.73%  [80.33%, 82.98%]
  Weighted F1   :  81.70%  [80.32%, 82.99%]
  Macro FPR     :   3.68%  [3.41%, 3.96%]
  Macro FNR     :  21.36%  [19.44%, 23.24%]


In [18]:
# Analysis 4: Grad-CAM  (patch_embed.proj — final spatial layer)

class GradCAM:
    def __init__(self, model):
        self._acts    = None
        self._grads   = None
        self._handles = []
        target = model.patch_embed.proj
        self._handles.append(target.register_forward_hook(self._fwd_hook))
        self._handles.append(target.register_full_backward_hook(self._bwd_hook))

    def _fwd_hook(self, module, inp, out):
        self._acts = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self._grads = grad_out[0].detach()

    def compute(self, model, img_t, class_idx=None):
        model.zero_grad()
        out = model(img_t)
        if class_idx is None:
            class_idx = int(out.argmax(1).item())
        out[0, class_idx].backward()
        weights = self._grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self._acts).sum(dim=1)).squeeze()
        cam = cam.cpu().float().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        prob = float(F.softmax(out.detach(), dim=1)[0, class_idx].item())
        return cam, class_idx, prob

    def remove(self):
        for h in self._handles:
            h.remove()


def upsample_cam(cam, size=224):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ).astype(np.float32) / 255.0


sample_per_class = {}
for i, path in enumerate(val_paths):
    lbl = int(val_labels[i])
    if lbl not in sample_per_class:
        sample_per_class[lbl] = (path, lbl)
    if len(sample_per_class) == NUM_CLASSES:
        break

val_tf  = get_val_transforms(config.img_size)
gcam    = GradCAM(eval_model)
eval_model.eval()

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 3, 6))

for col, cls_idx in enumerate(sorted(sample_per_class.keys())):
    path, lbl  = sample_per_class[cls_idx]
    raw_img    = Image.open(path).convert('RGB').resize((224, 224))
    img_t      = val_tf(raw_img).unsqueeze(0).to(device)
    cam, pred_idx, prob = gcam.compute(eval_model, img_t)
    cam_up     = upsample_cam(cam)
    axes[0, col].imshow(raw_img)
    axes[0, col].set_title(f'True: {CLASS_NAMES[lbl]}', fontsize=8)
    axes[0, col].axis('off')
    axes[1, col].imshow(raw_img, alpha=0.5)
    axes[1, col].imshow(cam_up, cmap='jet', alpha=0.5)
    axes[1, col].set_title(f'Pred: {CLASS_NAMES[pred_idx]}\n{prob*100:.1f}%', fontsize=8)
    axes[1, col].axis('off')

gcam.remove()
plt.suptitle('Grad-CAM: patch_embed.proj  (one sample per class)', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/gradcam_per_class.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved gradcam_per_class.png')


Saved gradcam_per_class.png


In [19]:
# Analysis 5: Saliency Faithfulness  (mask top-10% hot pixels, measure prob drop)

def faithfulness_test(model, n_samples=100, top_frac=0.10, seed=42):
    rng    = np.random.default_rng(seed)
    chosen = rng.choice(len(val_paths), min(n_samples, len(val_paths)), replace=False)
    tf     = get_val_transforms(config.img_size)
    gcam_f = GradCAM(model)
    model.eval()
    orig_p_list, masked_p_list = [], []

    for i in chosen:
        img_t = tf(Image.open(val_paths[i]).convert('RGB')).unsqueeze(0).to(device)
        cam, pred_idx, orig_p = gcam_f.compute(model, img_t)
        cam_up    = upsample_cam(cam, config.img_size)
        threshold = np.percentile(cam_up, (1 - top_frac) * 100)
        mask      = torch.from_numpy(
            (cam_up >= threshold).astype(np.float32)
        ).unsqueeze(0).unsqueeze(0).to(device)
        img_masked = img_t.detach() * (1.0 - mask)
        with torch.no_grad():
            out_m = model(img_masked)
        masked_p = float(F.softmax(out_m.float(), dim=1)[0, pred_idx].item())
        orig_p_list.append(orig_p)
        masked_p_list.append(masked_p)

    gcam_f.remove()
    return np.array(orig_p_list), np.array(masked_p_list)


orig_p, masked_p = faithfulness_test(eval_model, n_samples=100)
drops = orig_p - masked_p

print('Saliency Faithfulness  (mask top-10% hot pixels, n=100)')
print(f'  Mean original prob    : {orig_p.mean()*100:.2f}%')
print(f'  Mean masked prob      : {masked_p.mean()*100:.2f}%')
print(f'  Mean probability drop : {drops.mean()*100:.2f}%')
print(f'  Median prob drop      : {np.median(drops)*100:.2f}%')
print(f'  Samples with drop > 0 : {(drops > 0).sum()}/{len(drops)}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(drops * 100, bins=25, color='steelblue', edgecolor='white')
ax.axvline(drops.mean() * 100, color='red', linestyle='--',
           label=f'Mean drop: {drops.mean()*100:.1f}%')
ax.set_xlabel('Probability Drop (%)')
ax.set_ylabel('Count')
ax.set_title('Faithfulness: Prob Drop After Masking Top-10% Saliency Pixels')
ax.legend()
plt.tight_layout()
plt.savefig('vim_drone_rf_output/faithfulness_drop.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved faithfulness_drop.png')


Saliency Faithfulness  (mask top-10% hot pixels, n=100)
  Mean original prob    : 77.58%
  Mean masked prob      : 53.57%
  Mean probability drop : 24.02%
  Median prob drop      : 10.21%
  Samples with drop > 0 : 80/100
Saved faithfulness_drop.png


In [20]:
# Analysis 6: Noise Floor Rejection  (Grad-CAM entropy on background-class images)

def cam_entropy(cam_arr):
    flat = cam_arr.flatten().astype(np.float64) + 1e-9
    flat = flat / flat.sum()
    return float(-np.sum(flat * np.log(flat)))

noise_cls_idx   = CLASS_NAMES.index('Noise')
noise_idx_list  = [i for i in range(len(val_paths)) if val_labels[i] == noise_cls_idx]
signal_idx_list = [i for i in range(len(val_paths)) if val_labels[i] != noise_cls_idx]

tf     = get_val_transforms(config.img_size)
n_show = min(6, len(noise_idx_list))
eval_model.eval()

gcam_nr = GradCAM(eval_model)
noise_entropies = []
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6))

for col in range(n_show):
    path    = val_paths[noise_idx_list[col]]
    raw_img = Image.open(path).convert('RGB').resize((224, 224))
    img_t   = tf(raw_img).unsqueeze(0).to(device)
    cam, pred_idx, prob = gcam_nr.compute(eval_model, img_t, class_idx=noise_cls_idx)
    cam_up  = upsample_cam(cam)
    ent     = cam_entropy(cam_up)
    noise_entropies.append(ent)
    axes[0, col].imshow(raw_img)
    axes[0, col].set_title('Noise', fontsize=8)
    axes[0, col].axis('off')
    axes[1, col].imshow(raw_img, alpha=0.5)
    axes[1, col].imshow(cam_up, cmap='jet', alpha=0.5)
    axes[1, col].set_title(f'Pred: {CLASS_NAMES[pred_idx]}\nH={ent:.2f}', fontsize=8)
    axes[1, col].axis('off')

gcam_nr.remove()
plt.suptitle('Noise Floor Rejection: Grad-CAM on Background-Class Images', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/noise_floor_rejection.png', dpi=100, bbox_inches='tight')
plt.show()

gcam_sig = GradCAM(eval_model)
rng_sig  = np.random.default_rng(42)
sig_idx  = rng_sig.choice(len(signal_idx_list), min(60, len(signal_idx_list)), replace=False)
signal_entropies = []
for i in sig_idx:
    img_t = tf(Image.open(val_paths[signal_idx_list[i]]).convert('RGB')).unsqueeze(0).to(device)
    cam, _, _ = gcam_sig.compute(eval_model, img_t)
    cam_up    = upsample_cam(cam)
    signal_entropies.append(cam_entropy(cam_up))
gcam_sig.remove()

print(f'Mean Grad-CAM entropy  Noise class   : {np.mean(noise_entropies):.3f}')
print(f'Mean Grad-CAM entropy  Signal classes: {np.mean(signal_entropies):.3f}')
print(f'Higher entropy on noise = diffuse attribution (expected for background class)')


Mean Grad-CAM entropy  Noise class   : 10.244
Mean Grad-CAM entropy  Signal classes: 9.657
Higher entropy on noise = diffuse attribution (expected for background class)


In [21]:
# Analysis 7: Integrated Gradients  (input-space attribution, 50-step path)

def integrated_gradients(model, img_t, class_idx, n_steps=50, baseline=None):
    if baseline is None:
        baseline = torch.zeros_like(img_t)
    alphas     = torch.linspace(0.0, 1.0, n_steps, device=device)
    grad_sum   = torch.zeros_like(img_t)
    for alpha in alphas:
        interp   = baseline + alpha * (img_t - baseline)
        interp   = interp.detach().requires_grad_(True)
        out      = model(interp)
        score    = out[0, class_idx]
        score.backward()
        grad_sum += interp.grad.detach()
    ig = (img_t - baseline) * grad_sum / n_steps
    ig_map = ig.squeeze(0).abs().sum(dim=0).cpu().numpy()
    ig_map = (ig_map - ig_map.min()) / (ig_map.max() - ig_map.min() + 1e-8)
    return ig_map

eval_model.eval()
tf     = get_val_transforms(config.img_size)
n_show = min(NUM_CLASSES, len(sample_per_class))

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 3, 9))

for col, cls_idx in enumerate(sorted(sample_per_class.keys())):
    path, lbl  = sample_per_class[cls_idx]
    raw_img    = Image.open(path).convert('RGB').resize((224, 224))
    img_t      = val_tf(raw_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = eval_model(img_t)
    pred_idx = int(logits.argmax(1).item())
    prob     = float(F.softmax(logits.float(), dim=1)[0, pred_idx].item())

    ig_map = integrated_gradients(eval_model, img_t, pred_idx)

    gcam_ig = GradCAM(eval_model)
    cam, _, _ = gcam_ig.compute(eval_model, val_tf(raw_img).unsqueeze(0).to(device))
    gcam_ig.remove()
    cam_up = upsample_cam(cam)

    axes[0, col].imshow(raw_img)
    axes[0, col].set_title(f'True: {CLASS_NAMES[lbl]}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(raw_img, alpha=0.5)
    axes[1, col].imshow(ig_map, cmap='hot', alpha=0.6)
    axes[1, col].set_title(f'IntGrad\nPred: {CLASS_NAMES[pred_idx]}', fontsize=8)
    axes[1, col].axis('off')

    axes[2, col].imshow(raw_img, alpha=0.5)
    axes[2, col].imshow(cam_up, cmap='jet', alpha=0.5)
    axes[2, col].set_title(f'Grad-CAM\n{prob*100:.1f}%', fontsize=8)
    axes[2, col].axis('off')

plt.suptitle('Integrated Gradients vs Grad-CAM  (per class)', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/integrated_gradients.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved integrated_gradients.png')


Saved integrated_gradients.png


In [22]:
# Analysis 8: Occlusion Sensitivity  (patch-level sliding window)

def occlusion_sensitivity(model, img_t, class_idx, patch_size=16, stride=16):
    model.eval()
    _, _, H, W = img_t.shape
    with torch.no_grad():
        base_prob = float(F.softmax(model(img_t).float(), dim=1)[0, class_idx].item())

    n_h = (H - patch_size) // stride + 1
    n_w = (W - patch_size) // stride + 1
    occ_map = np.zeros((n_h, n_w), dtype=np.float32)

    for i in range(n_h):
        for j in range(n_w):
            r0, r1 = i * stride, i * stride + patch_size
            c0, c1 = j * stride, j * stride + patch_size
            masked  = img_t.clone()
            masked[:, :, r0:r1, c0:c1] = 0.0
            with torch.no_grad():
                occ_prob = float(
                    F.softmax(model(masked).float(), dim=1)[0, class_idx].item()
                )
            occ_map[i, j] = base_prob - occ_prob

    occ_map = (occ_map - occ_map.min()) / (occ_map.max() - occ_map.min() + 1e-8)
    return occ_map

tf     = get_val_transforms(config.img_size)
n_show = min(NUM_CLASSES, len(sample_per_class))

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6))

for col, cls_idx in enumerate(sorted(sample_per_class.keys())):
    path, lbl = sample_per_class[cls_idx]
    raw_img   = Image.open(path).convert('RGB').resize((224, 224))
    img_t     = tf(raw_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = eval_model(img_t)
    pred_idx = int(logits.argmax(1).item())

    occ_map = occlusion_sensitivity(eval_model, img_t, pred_idx,
                                    patch_size=16, stride=16)
    occ_up  = np.array(
        Image.fromarray((occ_map * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
    ).astype(np.float32) / 255.0

    axes[0, col].imshow(raw_img)
    axes[0, col].set_title(f'True: {CLASS_NAMES[lbl]}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(raw_img, alpha=0.5)
    axes[1, col].imshow(occ_up, cmap='RdYlGn', alpha=0.6)
    axes[1, col].set_title(f'Pred: {CLASS_NAMES[pred_idx]}', fontsize=8)
    axes[1, col].axis('off')

plt.suptitle('Occlusion Sensitivity: 16x16 patch stride  (green=important)', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/occlusion_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved occlusion_sensitivity.png')


Saved occlusion_sensitivity.png


In [23]:
# Analysis 9: SSM Hidden State Magnitude  (VisionMamba-specific)
# Hooks each VimBlock output and computes per-position L2 norm across the sequence.
# Positions 1..N map to image patches; position 0 is the CLS token.

def extract_block_activations(model, img_t):
    block_acts = []
    handles    = []

    def make_hook(buf):
        def hook(module, inp, out):
            buf.append(out.detach().float().cpu())
        return hook

    for layer in model.layers:
        buf = []
        block_acts.append(buf)
        handles.append(layer.register_forward_hook(make_hook(buf)))

    with torch.no_grad():
        with autocast(dtype=torch.bfloat16):
            _ = model(img_t)

    for h in handles:
        h.remove()

    result = []
    for buf in block_acts:
        if buf:
            out   = buf[0]                    # (1, N+1, D)
            norms = out.squeeze(0).norm(dim=-1).numpy()  # (N+1,)
            result.append(norms)
    return np.array(result)                   # (depth, N+1)

tf     = get_val_transforms(config.img_size)
n_show = min(NUM_CLASSES, len(sample_per_class))
H_patch = config.img_size // config.patch_size

fig, axes = plt.subplots(1, n_show, figsize=(n_show * 3.5, 4))

for col, cls_idx in enumerate(sorted(sample_per_class.keys())):
    path, lbl = sample_per_class[cls_idx]
    raw_img   = Image.open(path).convert('RGB').resize((224, 224))
    img_t     = tf(raw_img).unsqueeze(0).to(device)

    acts = extract_block_activations(eval_model, img_t)  # (depth, N+1)

    patch_acts = acts[:, 1:]                             # drop CLS
    mean_over_depth = patch_acts.mean(axis=0)            # (N,)
    spatial_map = mean_over_depth.reshape(H_patch, H_patch)
    spatial_map = (spatial_map - spatial_map.min()) / (spatial_map.max() - spatial_map.min() + 1e-8)
    spatial_up  = np.array(
        Image.fromarray((spatial_map * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
    ).astype(np.float32) / 255.0

    axes[col].imshow(raw_img, alpha=0.5)
    axes[col].imshow(spatial_up, cmap='magma', alpha=0.6)
    axes[col].set_title(f'{CLASS_NAMES[lbl]}', fontsize=8)
    axes[col].axis('off')

plt.suptitle('SSM Hidden State Magnitude: Mean Block Activation per Patch', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/ssm_state_magnitude.png', dpi=100, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
path, lbl = list(sample_per_class.values())[0]
img_t     = tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
acts      = extract_block_activations(eval_model, img_t)

im = ax2.imshow(acts, aspect='auto', cmap='viridis')
ax2.set_xlabel('Sequence Position (0=CLS, 1..N=patches)')
ax2.set_ylabel('VimBlock Depth')
ax2.set_title(f'SSM Activation Heatmap Across Depth  ({CLASS_NAMES[lbl]})')
plt.colorbar(im, ax=ax2, label='L2 Norm')
plt.tight_layout()
plt.savefig('vim_drone_rf_output/ssm_depth_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved ssm_state_magnitude.png and ssm_depth_heatmap.png')


Saved ssm_state_magnitude.png and ssm_depth_heatmap.png


In [24]:
# Analysis 10: Forward vs Backward SSM Scan Contribution  (VisionMamba-specific)
# Computes the L2 norm of each scan direction per patch position to reveal
# which scan (causal forward or reversed backward) drives each spatial region.

def fwd_bwd_contribution(model, img_t):
    fwd_norms_all = []
    bwd_norms_all = []
    handles       = []

    def make_block_hook(block, fwd_buf, bwd_buf):
        orig_forward = block.forward

        def patched_forward(x):
            residual = x
            h       = block.norm(x)
            y_fwd   = block.mamba_fwd(h)
            y_bwd   = block.mamba_bwd(h.flip([1])).flip([1])
            fwd_buf.append(y_fwd.detach().float().cpu())
            bwd_buf.append(y_bwd.detach().float().cpu())
            out = block.layer_scale(y_fwd + y_bwd)
            return residual + block.drop_path(out)

        return patched_forward

    orig_forwards = []
    fwd_bufs, bwd_bufs = [], []

    for block in model.layers:
        fwd_buf, bwd_buf = [], []
        fwd_bufs.append(fwd_buf)
        bwd_bufs.append(bwd_buf)
        orig_forwards.append(block.forward)
        block.forward = make_block_hook(block, fwd_buf, bwd_buf)

    with torch.no_grad():
        with autocast(dtype=torch.bfloat16):
            _ = model(img_t)

    for i, block in enumerate(model.layers):
        block.forward = orig_forwards[i]
        if fwd_bufs[i]:
            fwd_norms_all.append(fwd_bufs[i][0].squeeze(0).norm(dim=-1).numpy())
            bwd_norms_all.append(bwd_bufs[i][0].squeeze(0).norm(dim=-1).numpy())

    return np.array(fwd_norms_all), np.array(bwd_norms_all)  # (depth, N+1) each

tf         = get_val_transforms(config.img_size)
H_patch    = config.img_size // config.patch_size
eval_model.eval()
n_show     = min(NUM_CLASSES, len(sample_per_class))

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 3.5, 9))

for col, cls_idx in enumerate(sorted(sample_per_class.keys())):
    path, lbl = sample_per_class[cls_idx]
    raw_img   = Image.open(path).convert('RGB').resize((224, 224))
    img_t     = tf(raw_img).unsqueeze(0).to(device)

    fwd_norms, bwd_norms = fwd_bwd_contribution(eval_model, img_t)

    fwd_spatial = fwd_norms[:, 1:].mean(0).reshape(H_patch, H_patch)
    bwd_spatial = bwd_norms[:, 1:].mean(0).reshape(H_patch, H_patch)
    dom_map     = fwd_spatial - bwd_spatial

    def norm01(x):
        mn, mx = x.min(), x.max()
        return (x - mn) / (mx - mn + 1e-8)

    fwd_up = np.array(Image.fromarray((norm01(fwd_spatial) * 255).astype(np.uint8))
                      .resize((224, 224), Image.BILINEAR)).astype(np.float32) / 255.0
    bwd_up = np.array(Image.fromarray((norm01(bwd_spatial) * 255).astype(np.uint8))
                      .resize((224, 224), Image.BILINEAR)).astype(np.float32) / 255.0
    dom_up = np.array(Image.fromarray(
        (norm01(dom_map) * 255).astype(np.uint8)
    ).resize((224, 224), Image.BILINEAR)).astype(np.float32) / 255.0

    axes[0, col].imshow(raw_img, alpha=0.5)
    axes[0, col].imshow(fwd_up, cmap='Blues', alpha=0.6)
    axes[0, col].set_title(f'Fwd  {CLASS_NAMES[lbl]}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(raw_img, alpha=0.5)
    axes[1, col].imshow(bwd_up, cmap='Reds', alpha=0.6)
    axes[1, col].set_title('Bwd', fontsize=8)
    axes[1, col].axis('off')

    axes[2, col].imshow(raw_img, alpha=0.5)
    axes[2, col].imshow(dom_up, cmap='RdBu_r', alpha=0.6)
    axes[2, col].set_title('Fwd-Bwd dominance', fontsize=8)
    axes[2, col].axis('off')

plt.suptitle('Forward vs Backward SSM Scan Contribution  (blue=fwd, red=bwd)', fontsize=11)
plt.tight_layout()
plt.savefig('vim_drone_rf_output/fwd_bwd_scan.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved fwd_bwd_scan.png')


Saved fwd_bwd_scan.png


In [25]:
# Analysis 11: Digital Twin Waveform  (Grad-CAM attribution profiles by SNR)

tf              = get_val_transforms(config.img_size)
samples_per_snr = 10
eval_model.eval()

gcam_dt           = GradCAM(eval_model)
snr_freq_profiles = {}
snr_time_profiles = {}

for snr_val in ALL_SNR_LEVELS:
    snr_mask  = (val_snrs == snr_val)
    snr_paths = [val_paths[i] for i in range(len(val_paths)) if snr_mask[i]]
    if not snr_paths:
        continue
    rng  = np.random.default_rng(42)
    idxs = rng.choice(len(snr_paths), min(samples_per_snr, len(snr_paths)), replace=False)
    freq_acc, time_acc = [], []
    for i in idxs:
        img_t = tf(Image.open(snr_paths[i]).convert('RGB')).unsqueeze(0).to(device)
        cam, _, _ = gcam_dt.compute(eval_model, img_t)
        freq_acc.append(cam.mean(axis=1))
        time_acc.append(cam.mean(axis=0))
    snr_freq_profiles[snr_val] = np.mean(freq_acc, axis=0)
    snr_time_profiles[snr_val] = np.mean(time_acc, axis=0)

gcam_dt.remove()

n_levels = len(snr_freq_profiles)
cmap_dt  = plt.cm.RdYlGn
colors_dt = {s: cmap_dt(i / max(n_levels - 1, 1))
             for i, s in enumerate(sorted(snr_freq_profiles.keys()))}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for snr_val in sorted(snr_freq_profiles.keys()):
    c = colors_dt[snr_val]
    ax1.plot(snr_freq_profiles[snr_val], color=c, label=f'{snr_val:+d}dB', alpha=0.85, linewidth=1.2)
    ax2.plot(snr_time_profiles[snr_val], color=c, label=f'{snr_val:+d}dB', alpha=0.85, linewidth=1.2)

ax1.set_xlabel('Frequency Bin (patch index)')
ax1.set_ylabel('Mean Saliency')
ax1.set_title('Frequency Attribution Profile by SNR')
ax1.legend(fontsize=6, ncol=3, loc='upper right')

ax2.set_xlabel('Time Frame (patch index)')
ax2.set_ylabel('Mean Saliency')
ax2.set_title('Time Attribution Profile by SNR')
ax2.legend(fontsize=6, ncol=3, loc='upper right')

plt.tight_layout()
plt.savefig('vim_drone_rf_output/digital_twin_waveform.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved digital_twin_waveform.png')


Saved digital_twin_waveform.png


In [26]:
# Analysis 12: t-SNE Latent Space by SNR Level

from sklearn.manifold import TSNE

def extract_cls_features(model, max_samples=2000):
    rng = np.random.default_rng(42)
    n   = len(val_paths)
    if n > max_samples:
        chosen   = rng.choice(n, max_samples, replace=False)
        paths_s  = [val_paths[i] for i in chosen]
        labels_s = val_labels[chosen]
        snrs_s   = val_snrs[chosen]
    else:
        paths_s, labels_s, snrs_s = val_paths, val_labels.copy(), val_snrs.copy()

    feats   = []
    handle  = model.norm.register_forward_hook(
        lambda m, inp, out: feats.append(out[:, 0, :].float().detach().cpu())
    )
    ds     = DroneRFDataset(paths_s, labels_s, snrs_s,
                            transform=get_val_transforms(config.img_size))
    loader = DataLoader(ds, batch_size=config.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    model.eval()
    with torch.no_grad():
        for images, _ in loader:
            with autocast(dtype=torch.bfloat16):
                _ = model(images.to(device, non_blocking=True))
    handle.remove()
    return torch.cat(feats, dim=0).numpy(), labels_s, snrs_s

print('Extracting CLS token features...')
feat, feat_labels, feat_snrs = extract_cls_features(eval_model)
print(f'Features: {feat.shape}')

print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=40, learning_rate='auto',
            init='pca', random_state=42, n_iter=1000)
emb  = tsne.fit_transform(feat)

unique_snrs = sorted(set(feat_snrs.tolist()))
snr_norm    = plt.Normalize(vmin=min(unique_snrs), vmax=max(unique_snrs))
snr_cmap    = plt.cm.plasma
snr_color   = [snr_cmap(snr_norm(s)) for s in feat_snrs.tolist()]

cls_cmap   = plt.cm.tab10
cls_colors = [cls_cmap(i / NUM_CLASSES) for i in range(NUM_CLASSES)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.scatter(emb[:, 0], emb[:, 1], c=snr_color, s=8, alpha=0.7)
sm = plt.cm.ScalarMappable(cmap=snr_cmap, norm=snr_norm)
sm.set_array([])
plt.colorbar(sm, ax=ax1, label='SNR (dB)')
ax1.set_title('t-SNE: CLS Token  colored by SNR')
ax1.set_xlabel('t-SNE 1')
ax1.set_ylabel('t-SNE 2')

ax2.scatter(emb[:, 0], emb[:, 1],
            c=[cls_colors[l] for l in feat_labels.tolist()], s=8, alpha=0.7)
ax2.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=cls_colors[i], markersize=6, label=CLASS_NAMES[i])
    for i in range(NUM_CLASSES)
], fontsize=8)
ax2.set_title('t-SNE: CLS Token  colored by Class')
ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('vim_drone_rf_output/tsne_snr_class.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved tsne_snr_class.png')


Extracting CLS token features...
Features: (2000, 96)
Running t-SNE...
Saved tsne_snr_class.png


In [27]:
# Analysis 13: FLOPs and Inference Time

import time

def count_flops(model, input_size=(1, 3, 224, 224)):
    total   = [0]
    handles = []

    def linear_hook(m, inp, out):
        x = inp[0]
        if x.dim() == 2:
            total[0] += int(x.shape[0]) * int(x.shape[1]) * int(m.out_features) * 2
        elif x.dim() == 3:
            total[0] += int(x.shape[0]) * int(x.shape[1]) * int(x.shape[2]) * int(m.out_features) * 2

    def conv2d_hook(m, inp, out):
        b, c_out, h, w = out.shape
        k = m.kernel_size[0] * m.kernel_size[1]
        total[0] += int(b) * int(c_out) * int(h) * int(w) * (int(m.in_channels) // int(m.groups)) * k * 2

    def conv1d_hook(m, inp, out):
        b, c_out, l = out.shape
        total[0] += int(b) * int(c_out) * int(l) * (int(m.in_channels) // int(m.groups)) * int(m.kernel_size[0]) * 2

    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            handles.append(mod.register_forward_hook(linear_hook))
        elif isinstance(mod, nn.Conv2d):
            handles.append(mod.register_forward_hook(conv2d_hook))
        elif isinstance(mod, nn.Conv1d):
            handles.append(mod.register_forward_hook(conv1d_hook))

    x = torch.randn(*input_size, device=device)
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()

    return total[0], sum(p.numel() for p in model.parameters())

def measure_latency_ms(model, n_warmup=20, n_runs=100, batch_size=1):
    model.eval()
    x = torch.randn(batch_size, 3, 224, 224, device=device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000.0

flops, params = count_flops(eval_model)
lat_bs1  = measure_latency_ms(eval_model, batch_size=1)
lat_bs8  = measure_latency_ms(eval_model, batch_size=8)

print('VisionMamba-B Computational Cost')
print(f'  GFLOPs (batch=1)  : {flops/1e9:.3f}')
print(f'  Parameters        : {params/1e6:.3f} M')
print(f'  Latency bs=1      : {lat_bs1:.2f} ms  ({1000/lat_bs1:.1f} FPS)')
print(f'  Latency bs=8      : {lat_bs8/8:.2f} ms/img  ({8000/lat_bs8:.1f} FPS)')
print(f'  Acc (val)         : {(val_preds == true_labels).mean()*100:.2f}%')


VisionMamba-B Computational Cost
  GFLOPs (batch=1)  : 0.029
  Parameters        : 0.853 M
  Latency bs=1      : 6.59 ms  (151.7 FPS)
  Latency bs=8      : 3.87 ms/img  (258.4 FPS)
  Acc (val)         : 81.71%


In [28]:
# Analysis 14: Confusion Matrix  (per-model, per-class error breakdown)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, val_preds, labels=list(range(NUM_CLASSES)))

row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
cm_norm  = cm.astype(float) / row_sums

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, title in [
    (axes[0], cm,      'Raw Counts'),
    (axes[1], cm_norm, 'Row-Normalized'),
]:
    im  = ax.imshow(data, interpolation='nearest',
                    cmap='Blues', vmin=0, vmax=(1 if data is cm_norm else None))
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_title(f'VisionMamba-B  ({title})', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            val = cm_norm[i, j] if data is cm_norm else cm[i, j]
            fmt = f'{val:.2f}' if data is cm_norm else str(int(val))
            ax.text(j, i, fmt, ha='center', va='center', fontsize=7,
                    color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.savefig('vim_drone_rf_output/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print('Per-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc_c   = correct / total * 100.0 if total > 0 else 0.0
    print(f'  {name:<12}: {acc_c:.1f}%  ({correct}/{total})')
print('Saved confusion_matrix.png')


Per-class accuracy:
  DJI         : 69.9%  (179/256)
  FutabaT14   : 73.3%  (509/694)
  FutabaT7    : 76.9%  (123/160)
  Graupner    : 59.4%  (95/160)
  Noise       : 86.2%  (1530/1775)
  Taranis     : 91.3%  (304/333)
  Turnigy     : 93.6%  (160/171)
Saved confusion_matrix.png
